# Finalize Results
Builds the evaluation bundle from your feat-extract + judge outputs.

**Inputs** (produced by `run_feat_extract.sh` + `run_judge.sh`):
- `../original_nb_data/MICE_Output/results_summary_judged.csv`
- `../original_nb_data/MICE_Output/results_layers.csv`

**Outputs** (written into `../original_nb_data/MICE_Output/<BUNDLE_DIR>/`):
- `summary_full.csv` - copy of the judged summary, eval-canonical filename
- `layers_full.csv` - copy of the layers CSV, eval-canonical filename
- `data_bundle.joblib` — train/val/test split (60/20/20, stratified by `type` x `judge_decision`, `random_state=42`)

The default `BUNDLE_DIR = "my-eval"` so this never overwrites the precomputed `eval-1k/` bundle.

In [ ]:
import os
import numpy as np
import pandas as pd
import joblib
from pathlib import Path
from sklearn.model_selection import train_test_split

In [ ]:
# Papermill parameters — override via `-p NAME VALUE`.
BUNDLE_DIR = "my-eval"  # output subdir under MICE_Output/; do NOT default to 'eval-1k' (precomputed bundle)

In [ ]:
if BUNDLE_DIR == "eval-1k":
    print("IMPORTANT: you are overwriting our precomputed data, please make sure this is intended! If it's intended, remove the raise SystemExit in the line below")
    raise SystemExit

In [ ]:
# paths
BASE_PATH = Path.cwd().parent
ORIGIN_PATH = f"{BASE_PATH}/original_nb_data/MICE_Output"
OUTPUT_PATH = f"{ORIGIN_PATH}/{BUNDLE_DIR}"
os.makedirs(OUTPUT_PATH, exist_ok=True)

# inputs from the local feat-extract + judge pipeline
summary_in = os.path.join(ORIGIN_PATH, "results_summary_judged.csv")
layers_in  = os.path.join(ORIGIN_PATH, "results_layers.csv")

# outputs (eval-canonical filenames)
summary_full_path = os.path.join(OUTPUT_PATH, "summary_full.csv")
layers_full_path  = os.path.join(OUTPUT_PATH, "layers_full.csv")
bundle_path       = os.path.join(OUTPUT_PATH, "data_bundle.joblib")

print(f"Reading from {ORIGIN_PATH}")
print(f"Writing to   {OUTPUT_PATH}")

In [ ]:
# load and copy under eval-canonical filenames
summary_df = pd.read_csv(summary_in)
layers_df  = pd.read_csv(layers_in)

summary_df.to_csv(summary_full_path, index=False)
layers_df.to_csv(layers_full_path,   index=False)

print(f"summary rows: {len(summary_df)}, layers rows: {len(layers_df)}")

In [ ]:
# pivot layers wide and merge with summary
layer_pivot = (
    layers_df.pivot_table(index="question_id", columns="layer", values="f1")
    .sort_index(axis=1)
    .reset_index()
)
layer_pivot.columns = ["question_id"] + [f"f1_layer_{c}" for c in layer_pivot.columns[1:]]

df = summary_df.merge(layer_pivot, on="question_id")

# feature definition
layer_cols = [c for c in df.columns if c.startswith("f1_layer_")]
feature_cols = layer_cols + ["normalized_log_confidence"]

# stratify on type AND judge_decision so both label and outcome balance are preserved
df["stratify_key"] = df["type"].astype(str) + "_" + df["judge_decision"].astype(str)

# 60 / 20 / 20 stratified split (seed = 42 to match eval-1k bundle)
df_tv, df_test = train_test_split(
    df, test_size=0.20, random_state=42, stratify=df["stratify_key"]
)
df_train, df_val = train_test_split(
    df_tv, test_size=0.25, random_state=42, stratify=df_tv["stratify_key"]
)

def get_xyt(target_df):
    X = target_df[feature_cols].values.astype(np.float32)
    y = target_df["judge_decision"].values.astype(int)
    act_type  = target_df["type"].values.astype(str)
    pred_type = target_df["predicted_type"].values.astype(str)
    return X, y, act_type, pred_type

X_train, y_train, type_train, ptype_train = get_xyt(df_train)
X_val,   y_val,   type_val,   ptype_val   = get_xyt(df_val)
X_test,  y_test,  type_test,  ptype_test  = get_xyt(df_test)

data_bundle = {
    "train": (X_train, y_train, type_train, ptype_train),
    "val":   (X_val,   y_val,   type_val,   ptype_val),
    "test":  (X_test,  y_test,  type_test,  ptype_test),
    "feature_names": feature_cols,
    "metadata": {
        "train_ids": df_train["question_id"].values,
        "val_ids":   df_val["question_id"].values,
        "test_ids":  df_test["question_id"].values,
    },
}

joblib.dump(data_bundle, bundle_path)

print(f"Saved bundle: {bundle_path}")
print(f"Train: {len(X_train)}  Val: {len(X_val)}  Test: {len(X_test)}")
print(f"Features: {len(feature_cols)} ({len(layer_cols)} layers + 1 confidence)")